In [ ]:
import pandas as pd
import re
import mysql.connector
from tqdm import tqdm
from unidecode import unidecode

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz

THRESHOLD_TOKEN = 95
THRESHOLD_CHAR = 75
EMBED_THRESHOLD = 0.82

# TRACKING

stats = {}
embedding_cases = []

conn = mysql.connector.connect(
    host='localhost',
    port=3307,
    user='root',
    password='root',
    database='chuyen_doi'
)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS hashtag_sbert")

cursor.execute("""
CREATE TABLE hashtag_sbert (
    id INT AUTO_INCREMENT PRIMARY KEY,
    cvsh_id INT,
    org_hashtag TEXT,
    hashtag TEXT,
    root_id INT,
    root_amount INT
)
""")
conn.commit()

df = pd.read_sql("""
SELECT cvsh_id, cvsh_hashtag, cvsh_amount
FROM hashtag_top5
""", conn)

print("Initial:", len(df))

# PREPROCESS 

def normalize(text):
    text = str(text).lower().strip()
    text = unidecode(text) 
    text = re.sub(r'[^\w\s]', ' ', text)
    return " ".join(text.split())

def normalize_raw(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)
    return " ".join(text.split())

df['clean_text'] = df['cvsh_hashtag'].apply(normalize)
df['raw_text'] = df['cvsh_hashtag'].apply(normalize_raw)
df['org_hashtag'] = df['cvsh_hashtag']

df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))
df = df[df['word_count'] > 1].reset_index(drop=True)

df = df.sort_values('cvsh_amount', ascending=False).reset_index(drop=True)

print("After filter:", len(df))

# SBERT 

print("Loading SBERT...")
model = SentenceTransformer("keepitreal/vietnamese-sbert")

embeddings = model.encode(
    df['raw_text'].tolist(),
    show_progress_bar=True,
    batch_size=64
)

df['idx'] = range(len(df))
embedding_map = dict(zip(df['idx'], embeddings))

# RULE HELPERS

NEGATION = {"khong", "chua", "ko", "kh"}

def has_negation(tokens):
    return any(t in NEGATION for t in tokens)

def extract_entities(text):
    numbers = re.findall(r'\d+', text)
    units = re.findall(r'(thang|nam|ngay|lan|buoi|%|tr|k|co so|nhan vien|giuong)', text)
    return numbers, units

# SIMILARITY FUNCTION

def is_similar(i, j):

    a_clean = df.loc[i, 'clean_text']
    b_clean = df.loc[j, 'clean_text']

    a_raw = df.loc[i, 'raw_text']
    b_raw = df.loc[j, 'raw_text']

    ta = a_clean.split()
    tb = b_clean.split()

    # NEGATION
    if has_negation(ta) != has_negation(tb):
        return False, "negation"

    # RULE
    na, ua = extract_entities(a_clean)
    nb, ub = extract_entities(b_clean)

    if na != nb:
        return False, "number"

    if na and ua != ub:
        return False, "unit"

    # token overlap
    sa, sb = set(ta), set(tb)
    if len(sa & sb) / max(len(sa), len(sb)) < 0.35:
        return False, "token_low"
    diff_a = sa - sb
    diff_b = sb - sa

    if len(diff_a) == 1 and len(diff_b) == 1:
        return False, "diff_token"

    # FUZZY
    token_score = fuzz.token_set_ratio(a_clean, b_clean)
    char_score = fuzz.ratio(a_clean, b_clean)

    if token_score >= THRESHOLD_TOKEN:
        return True, "fuzzy_strong"

    # EMBEDDING
    emb_a = embedding_map.get(i)
    emb_b = embedding_map.get(j)

    if emb_a is not None and emb_b is not None:
        sim = cosine_similarity([emb_a], [emb_b])[0][0]

        if sim >= EMBED_THRESHOLD:
            # embedding_cases.append((a_raw, b_raw, sim))
            embedding_cases.append((
            df.loc[i, 'cvsh_id'],
            df.loc[j, 'cvsh_id'],
            a_raw,
            b_raw,
            sim
            ))
            return True, "embedding"

    if token_score >= 85 and char_score >= THRESHOLD_CHAR:
        return True, "fuzzy_medium"

    return False, "no_match"

# CLUSTERING

clusters = []

print("Clustering...")

for i in tqdm(range(len(df))):
    assigned = False

    for cluster in clusters:
        for j in cluster['samples'][:3]:
            match, reason = is_similar(i, j)

            if match:
                cluster['members'].append(i)
                cluster['samples'].append(i)

                stats[reason] = stats.get(reason, 0) + 1

                assigned = True
                break

        if assigned:
            break

    if not assigned:
        clusters.append({
            "root": i,
            "members": [i],
            "samples": [i]
        })

print("Clusters:", len(clusters))

# MAP CLUSTER

cluster_map = {}

for cid, c in enumerate(clusters):
    for idx in c['members']:
        cluster_map[idx] = cid

df['cluster_id'] = df['idx'].map(cluster_map)

# ROOT SELECTION

root_df = (
    df.sort_values('cvsh_amount', ascending=False)
      .groupby('cluster_id', as_index=False)
      .first()[['cluster_id', 'cvsh_id', 'cvsh_amount', 'org_hashtag']]
      .rename(columns={
          'cvsh_id': 'root_id',
          'cvsh_amount': 'root_amount',
          'org_hashtag': 'root_hashtag'
      })
)

df = df.merge(root_df, on='cluster_id', how='left')
df['hashtag'] = df['root_hashtag']

result = df[[
    'cvsh_id',
    'org_hashtag',
    'hashtag',
    'root_id',
    'root_amount'
]].drop_duplicates(subset=['cvsh_id'])

print("Final rows:", len(result))

# INSERT DB

data = [
    (
        int(r.cvsh_id),
        r.org_hashtag,
        r.hashtag,
        int(r.root_id),
        int(r.root_amount)
    )
    for _, r in result.iterrows()
]

cursor.executemany("""
INSERT INTO hashtag_sbert
(cvsh_id, org_hashtag, hashtag, root_id, root_amount)
VALUES (%s, %s, %s, %s, %s)
""", data)

conn.commit()

print("DONE")

# EVALUATION

print("\n====================")
print("EVALUATION")
print("====================")

total = len(df)
cluster_sizes = [len(c['members']) for c in clusters]

print("Total hashtags:", total)
print("Total clusters:", len(clusters))
# print("Compression ratio:", round(total / len(clusters), 2))
# print("Singleton clusters:", sum(1 for c in clusters if len(c['members']) == 1))

# MATCH STATS

print("\nMATCH STATS:")
for k, v in stats.items():
    print(k, ":", v)

# EMBEDDING ANALYSIS

print("\nEmbedding cases:", len(embedding_cases))

if embedding_cases:
    emb_df = pd.DataFrame(
    embedding_cases,
    columns=["cvsh_id_a", "cvsh_id_b", "text_a", "text_b", "similarity"]
    )

    print("\nEmbedding stats:")
    print("Mean:", emb_df["similarity"].mean())
    print("Max:", emb_df["similarity"].max())
    print("Min:", emb_df["similarity"].min())

    print("\nTop matches:")
    print(emb_df.sort_values("similarity", ascending=False).head(10))

    emb_df.to_csv("embedding_cases_sbert.csv", index=False)

cursor.close()
conn.close()